# Bangla RAG question answering with a fine-tuned BanglaT5: Colab runner

Runtime: **Runtime > Change runtime type > L4 GPU** (High-RAM optional).
Run the cells top to bottom. Outputs and checkpoints are saved to Google Drive so a disconnect loses nothing.

**Status:** Step 0 (setup and token-length check) is ready. Training and evaluation cells are added in later steps.

## 1. Get the code and connect Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/mehediparvez/bangla_rag_evaluation.git"
BRANCH = "bangla-t5-rag-qa"
REPO_DIR = "/content/bangla_rag_evaluation"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}
%cd {REPO_DIR}

# Everything worth keeping goes here (survives a Colab disconnect)
OUT_DIR = "/content/drive/MyDrive/bangla_t5_rag"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving outputs to", OUT_DIR)

## 2. Install dependencies and check the GPU

In [ ]:
!pip -q install transformers sentencepiece
!pip -q install git+https://github.com/csebuetnlp/normalizer
!nvidia-smi

## 3. Download BanglaRQA
The dataset is fetched at run time and is not stored in the repository (CC BY-NC-SA 4.0).

In [ ]:
!python -m bangla_rag.data

## 4. Step 0: are whole passages too long for BanglaT5?
BanglaT5 reads at most 512 tokens. This counts how many passages are longer.
Send the printed table back before going further: it decides whether whole passages can be the retrieval unit.

In [ ]:
!python scripts/check_token_lengths.py

## 5. Pre-flight: speed and precision check (about 10 minutes)
Runs 150 optimizer steps twice, in bf16 and in fp32, and does not save anything.
Send back both `SPEED TEST` lines. We want: no NaN, similar loss in both, and the examples/second, which sets how many epochs are affordable.

In [ ]:
!python scripts/train_t5.py --recipe fast --precision bf16 --max-steps 150 --out-dir /content/preflight_bf16
!python scripts/train_t5.py --recipe fast --precision fp32 --max-steps 150 --out-dir /content/preflight_fp32

## 6. Train
Pick ONE recipe. Re-run the same cell after a disconnect: `--resume` continues from the last finished epoch (a partly finished epoch restarts).

* `fast`: 10 epochs, effective batch 16, lr 1e-4 (cheaper, my own recipe)
* `authors`: 15 epochs, batch 2, lr 5e-5 (the dataset authors' recipe; slower, most comparable to their scores)

Checkpoints, `log.jsonl` and `training_curve.png` go to Google Drive.

In [ ]:
!python scripts/train_t5.py --recipe fast --resume --out-dir {OUT_DIR}/run_fast

In [ ]:
# Only if you decide to run the authors' recipe instead:
# !python scripts/train_t5.py --recipe authors --resume --out-dir {OUT_DIR}/run_authors

## 7. Evaluate on the test set: gold passage
Runs the best checkpoint (chosen on validation) on all 1,493 test questions with the question's own passage, in fp32 like the dataset authors.
Takes a few minutes. Compare the printed numbers with the published ones (overall EM 62.42, F1 78.11).
Writes `gold_preds.jsonl` and `gold_summary.json` next to the model on Drive.

In [ ]:
!python scripts/eval_t5.py --model-dir {OUT_DIR}/run_fast/best --condition gold --out-dir {OUT_DIR}/run_fast/eval

## Next steps (cells added later)
8. Passage-level BM25, dense and hybrid retrievers, hit rates
9. Evaluate the three retrieval conditions (and the no-passage extra) with `--condition retrieved`
10. McNemar tests with Holm correction, bootstrap intervals, error-analysis sample